## 0 · Install dependencies

In [ ]:
!pip install pmdarima lightgbm xgboost -q

## 1 · Imports & Configuration

In [ ]:
import os, warnings
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model    import LinearRegression, Ridge
from sklearn.preprocessing   import StandardScaler
from sklearn.metrics         import mean_absolute_error, mean_squared_error
import xgboost  as xgb
import lightgbm as lgb

warnings.filterwarnings('ignore')

TARGET_COL = "Active Users"
YLABEL     = "Active Users"
CELL_NAME  = "ZAG_0158_C01_125_l3"
PLOT_DIR   = "active_users_plots"
TABLE_PATH = "comparison_table_ActiveUsers.csv"
HORIZONS   = [6, 24]
os.makedirs(PLOT_DIR, exist_ok=True)
print("Config ready — output dir:", PLOT_DIR)
print("Target KPI  :", TARGET_COL)

## 2 · Data Loading & Feature Engineering

In [ ]:
df = pd.read_csv("../datasets/final_kpi_data.csv")
df = df[['Time', 'Cell Name', 'Radio Site', 'Region',
         'Active Users', 'LTE DL User Throughput', 'DL PRB Load',
         'Cell FDD TDD Indication']]
df['Time'] = pd.to_datetime(df['Time'])
df = df.sort_values(['Cell Name', 'Time'])

# ── Identity encodings ──────────────────────────────────────────────────
# cell_id  : unique per cell (finest granularity)
# site_id  : unique per Radio Site  (groups 2-3 cells sharing the same mast)
# region_id: unique per Region      (groups all sites in the same area)
# Encoding as integer category codes keeps memory low and is compatible
# with tree-based models directly.
df['cell_id']   = df['Cell Name'].astype('category').cat.codes
df['site_id']   = df['Radio Site'].astype('category').cat.codes
df['region_id'] = df['Region'].astype('category').cat.codes

# ── FDD/TDD one-hot encoding
df = pd.get_dummies(df, columns=['Cell FDD TDD Indication'])

# ── Lag constants
LAG_2W, LAG_1W, LAG_1D = 24*14, 24*7, 24

# ── 2-week lags (all three KPIs)
df['prb_lag_2w']        = df.groupby('Cell Name')['DL PRB Load'].shift(LAG_2W)
df['throughput_lag_2w'] = df.groupby('Cell Name')['LTE DL User Throughput'].shift(LAG_2W)
df['users_lag_2w']      = df.groupby('Cell Name')['Active Users'].shift(LAG_2W)

# ── 1-week lags
df['prb_lag_1w']        = df.groupby('Cell Name')['DL PRB Load'].shift(LAG_1W)
df['throughput_lag_1w'] = df.groupby('Cell Name')['LTE DL User Throughput'].shift(LAG_1W)
df['users_lag_1w']      = df.groupby('Cell Name')['Active Users'].shift(LAG_1W)

# ── 24-hour lags
df['prb_lag_1d']        = df.groupby('Cell Name')['DL PRB Load'].shift(LAG_1D)
df['throughput_lag_1d'] = df.groupby('Cell Name')['LTE DL User Throughput'].shift(LAG_1D)
df['users_lag_1d']      = df.groupby('Cell Name')['Active Users'].shift(LAG_1D)

# ── Rolling 24h mean for target KPI (shift 1 to avoid leakage)
df['users_rolling_24h'] = (
    df.groupby('Cell Name')['Active Users']
    .transform(lambda x: x.shift(1).rolling(24, min_periods=1).mean())
)

# ── Cyclical time encoding
df['hour_sin']   = np.sin(2*np.pi*df['Time'].dt.hour       / 24)
df['hour_cos']   = np.cos(2*np.pi*df['Time'].dt.hour       / 24)
df['dow_sin']    = np.sin(2*np.pi*df['Time'].dt.dayofweek  / 7)
df['dow_cos']    = np.cos(2*np.pi*df['Time'].dt.dayofweek  / 7)
df['month_sin']  = np.sin(2*np.pi*df['Time'].dt.month      / 12)
df['month_cos']  = np.cos(2*np.pi*df['Time'].dt.month      / 12)
df['is_weekend'] = (df['Time'].dt.dayofweek >= 5).astype(int)

df = df.ffill().bfill()

# ── Feature list (excludes current-hour value of the target KPI)
BASE_FEATURES = [
    # ── Cell / site / region identity
    'cell_id', 'site_id', 'region_id',
    # ── 24-hour lags (all three KPIs)
    'prb_lag_1d', 'throughput_lag_1d', 'users_lag_1d',
    # ── 1-week lags
    'prb_lag_1w', 'throughput_lag_1w', 'users_lag_1w',
    # ── 2-week lags
    'prb_lag_2w', 'throughput_lag_2w', 'users_lag_2w',
    # ── Rolling 24h mean for target KPI
    'users_rolling_24h',
    # ── Cyclical time features
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    'month_sin', 'month_cos', 'is_weekend',
] + [c for c in df.columns if 'Cell FDD TDD Indication_' in c]

print(f"Rows: {len(df):,}   Features: {len(BASE_FEATURES)}")

## 3 · Evaluation Metrics (MAE / RMSE / MAPE)

In [ ]:
def evaluate(y_true, y_pred):
    """Returns (MAE, RMSE, MAPE). Ignores zero-valued samples in MAPE."""
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    return round(mae,4), round(rmse,4), round(mape,4)

## 4 · Train / Test Split (80 / 20 chronological)

In [ ]:
split_date = df['Time'].quantile(0.8)
train_df   = df[df['Time'] <= split_date].copy()
test_df    = df[df['Time'] >  split_date].copy()
print(f"Train: {len(train_df):,}   Test: {len(test_df):,}   Split: {split_date}")

## 5 · ARIMA
> ARIMA supports only univariate time series.  
> Trained & evaluated exclusively on the pilot cell for `Active Users`.

In [ ]:
from pmdarima    import auto_arima
import statsmodels.api as sm

cell_df  = df[df['Cell Name'] == CELL_NAME].sort_values('Time').reset_index(drop=True)
n_split  = int(len(cell_df) * 0.8)
arima_tr = cell_df['Active Users'].iloc[:n_split]
arima_te = cell_df['Active Users'].iloc[n_split:]

print('Running auto_arima to find order …')
auto_mdl = auto_arima(arima_tr[:500], seasonal=False, stepwise=True,
                      suppress_warnings=True, error_action='ignore')
ORDER    = auto_mdl.order
ORDER    = (10,1,10)

print(f'Best ARIMA order: {ORDER}')

In [ ]:
def arima_walk_forward(train_series, test_series, horizon):
    history  = list(train_series)
    preds, actuals = [], []
    test_arr = test_series.values
    for i in range(0, len(test_arr) - horizon, horizon):
        mdl = sm.tsa.ARIMA(history, order=ORDER)
        fit = mdl.fit()
        fc  = fit.forecast(steps=horizon)
        preds.append(float(fc[-1]))
        actuals.append(float(test_arr[i + horizon - 1]))
        history.extend(test_arr[i:i+horizon].tolist())
    return np.array(actuals), np.array(preds)

print('ARIMA H6  walk-forward …')
act_h6,  pred_h6  = arima_walk_forward(arima_tr, arima_te, 6)
mae6, rmse6, mape6 = evaluate(act_h6, pred_h6)
print(f'  H6  → MAE={mae6}  RMSE={rmse6}  MAPE={mape6}%')

print('ARIMA H24 walk-forward …')
act_h24, pred_h24  = arima_walk_forward(arima_tr, arima_te, 24)
mae24, rmse24, mape24 = evaluate(act_h24, pred_h24)
print(f'  H24 → MAE={mae24}  RMSE={rmse24}  MAPE={mape24}%')

arima_metrics = {
    'H6 MAE':mae6,  'H6 RMSE':rmse6,  'H6 MAPE':mape6,
    'H24 MAE':mae24,'H24 RMSE':rmse24,'H24 MAPE':mape24
}

## 6 · Prepare Direct-Forecast Targets (H6 & H24)

In [ ]:
results = {}

def make_horizon_dataset(horizon):
    """Shift target KPI forward by `horizon` steps per cell for direct forecasting."""
    d = df.copy()
    d['target'] = d.groupby('Cell Name')['Active Users'].shift(-horizon)
    d = d.dropna(subset=['target'])
    return d[d['Time'] <= split_date], d[d['Time'] > split_date]

print('Horizon datasets ready.')

## 7 · ML Models — Linear Regression, Ridge, XGBoost, LightGBM
> **Random Forest excluded**: requires >1 GB contiguous RAM on 10M rows (MemoryError confirmed during initial testing).

In [ ]:
import copy

ML_MODELS = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression':  Ridge(alpha=1.0),
    'XGBoost':  xgb.XGBRegressor(n_estimators=100, random_state=42,
                                  eval_metric='rmse', verbosity=0),
    'LightGBM': lgb.LGBMRegressor(n_estimators=100, random_state=42, verbose=-1),
}

ml_preds_h24  = {}    # model_name → H24 predictions on pilot cell test rows
xgb_model_h24 = None  # saved for feature importance
xgb_scaler    = None

for name, base_model in ML_MODELS.items():
    print(f"\n{'='*52}\n  {name}")
    m = {}
    for H in HORIZONS:
        tr_h, te_h = make_horizon_dataset(H)
        X_tr = tr_h[BASE_FEATURES].values;  y_tr = tr_h['target'].values
        X_te = te_h[BASE_FEATURES].values;  y_te = te_h['target'].values

        scaler  = StandardScaler()
        X_tr_s  = scaler.fit_transform(X_tr)
        X_te_s  = scaler.transform(X_te)

        model = copy.deepcopy(base_model)
        model.fit(X_tr_s, y_tr)
        preds = model.predict(X_te_s)

        mae, rmse, mape = evaluate(y_te, preds)
        m[f'H{H} MAE']  = mae
        m[f'H{H} RMSE'] = rmse
        m[f'H{H} MAPE'] = mape
        print(f"  H{H:2d} → MAE={mae:.4f}  RMSE={rmse:.4f}  MAPE={mape:.2f}%")

        if H == 24:
            zag_te  = te_h[te_h['Cell Name'] == CELL_NAME].sort_values('Time')
            zag_X   = scaler.transform(zag_te[BASE_FEATURES].values)
            ml_preds_h24[name] = model.predict(zag_X)
            if name == 'XGBoost':
                xgb_model_h24 = model
                xgb_scaler    = scaler

    results[name] = m

print("\nAll ML models done.")

## 8 · Neural Network

In [ ]:
import tensorflow as tf
from keras.models    import Sequential
from keras.layers    import Dense, Dropout
from keras.callbacks import EarlyStopping
import keras.backend as K

nn_metrics   = {}
nn_preds_h24 = None

for H in HORIZONS:
    print(f"\nNeural Network H{H} …")
    K.clear_session()

    tr_h, te_h = make_horizon_dataset(H)
    X_tr = tr_h[BASE_FEATURES].values;  y_tr = tr_h['target'].values
    X_te = te_h[BASE_FEATURES].values;  y_te = te_h['target'].values

    scaler  = StandardScaler()
    X_tr_s  = scaler.fit_transform(X_tr)
    X_te_s  = scaler.transform(X_te)

    model = Sequential([
        Dense(128, activation='relu', input_shape=(X_tr_s.shape[1],)),
        Dropout(0.2),
        Dense(64,  activation='relu'),
        Dropout(0.2),
        Dense(32,  activation='relu'),
        Dropout(0.2),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    es = EarlyStopping(patience=3, restore_best_weights=True, verbose=0)
    model.fit(X_tr_s, y_tr, epochs=20, batch_size=512,
              validation_split=0.1, callbacks=[es], verbose=0)

    preds = model.predict(X_te_s, verbose=0).flatten()
    mae, rmse, mape = evaluate(y_te, preds)
    nn_metrics[f'H{H} MAE']  = mae
    nn_metrics[f'H{H} RMSE'] = rmse
    nn_metrics[f'H{H} MAPE'] = mape
    print(f"  H{H:2d} → MAE={mae:.4f}  RMSE={rmse:.4f}  MAPE={mape:.2f}%")

    if H == 24:
        zag_te       = te_h[te_h['Cell Name'] == CELL_NAME].sort_values('Time')
        zag_X        = scaler.transform(zag_te[BASE_FEATURES].values)
        nn_preds_h24 = model.predict(zag_X, verbose=0).flatten()

results['Neural Network'] = nn_metrics
print("Neural Network done.")

## 9 · Comparison Table

In [ ]:
results['ARIMA'] = arima_metrics

ROW_ORDER = ['Linear Regression','Ridge Regression','ARIMA',
             'XGBoost','LightGBM','Neural Network']
COLS = ['H6 MAE','H6 RMSE','H6 MAPE','H24 MAE','H24 RMSE','H24 MAPE']

table = pd.DataFrame({m: results[m] for m in ROW_ORDER}).T[COLS]
table.index.name = 'Model'
print(table.to_string())
table.to_csv(TABLE_PATH)
print(f"\nSaved → {TABLE_PATH}")

## 10 · Forecast Plots  (48h history + 24h prediction vs actual)

In [ ]:
def forecast_plot(model_name, times_72, actual_72, pred_24, save_path):
    t_hist, t_fore = times_72.iloc[:48], times_72.iloc[48:]
    a_hist, a_fore = actual_72[:48],       actual_72[48:]

    fig, ax = plt.subplots(figsize=(15, 5))
    ax.plot(t_hist, a_hist, color='steelblue', linewidth=2,
            label='Actual (history 48h)')
    ax.plot(t_fore, a_fore, color='steelblue', linewidth=2, alpha=0.4,
            label='Actual (forecast window)')
    ax.plot(t_fore, pred_24, color='tomato', linewidth=2, linestyle='--',
            label='Predicted (24h ahead)')
    ax.axvspan(t_fore.iloc[0], t_fore.iloc[-1], alpha=0.07, color='orange')
    ax.axvline(x=t_fore.iloc[0], color='gray', linestyle=':', linewidth=1.5)
    ax.set_title(f"{model_name}  |  {CELL_NAME}", fontsize=13, fontweight='bold')
    ax.set_xlabel('Time')
    ax.set_ylabel(YLABEL)
    ax.legend(loc='upper left')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()
    print(f'  Saved → {save_path}')

### 11a · ARIMA forecast plot

In [ ]:
test_cell      = cell_df.iloc[n_split:].reset_index(drop=True)
window72       = test_cell.iloc[-72:].reset_index(drop=True)
boundary_idx   = n_split + len(test_cell) - 24
arima_fit_data = cell_df['Active Users'].iloc[:boundary_idx]
fit_plot       = sm.tsa.ARIMA(arima_fit_data, order=ORDER).fit()
arima_fc_24    = fit_plot.forecast(steps=24).values

forecast_plot(
    model_name = 'ARIMA',
    times_72   = window72['Time'],
    actual_72  = window72['Active Users'].values,
    pred_24    = arima_fc_24,
    save_path  = os.path.join(PLOT_DIR, 'ARIMA_Active_Users.png')
)

### 11b · ML & NN forecast plots

In [ ]:
_, te_h24    = make_horizon_dataset(24)
zag_full     = te_h24[te_h24['Cell Name'] == CELL_NAME].sort_values('Time').reset_index(drop=True)
win72        = zag_full.iloc[-72:].reset_index(drop=True)
times_72_ml  = win72['Time']
actual_72_ml = win72['Active Users'].values
offset       = len(zag_full) - 72

for name in ['Linear Regression', 'Ridge Regression', 'XGBoost', 'LightGBM']:
    pred_24h  = ml_preds_h24[name][offset+24 : offset+48]
    safe_name = name.replace(' ', '_')
    forecast_plot(
        model_name = name,
        times_72   = times_72_ml,
        actual_72  = actual_72_ml,
        pred_24    = pred_24h,
        save_path  = os.path.join(PLOT_DIR, f'{safe_name}_Active_Users.png')
    )

if nn_preds_h24 is not None:
    forecast_plot(
        model_name = 'Neural Network',
        times_72   = times_72_ml,
        actual_72  = actual_72_ml,
        pred_24    = nn_preds_h24[offset+24 : offset+48],
        save_path  = os.path.join(PLOT_DIR, f'Neural_Network_Active_Users.png')
    )

print('All forecast plots saved to:', PLOT_DIR)

## 12 · XGBoost Feature Importance (H+24)

In [ ]:
if xgb_model_h24 is not None:
    importances = xgb_model_h24.feature_importances_
    # Sort descending
    idx         = np.argsort(importances)[::-1]
    sorted_feats = [BASE_FEATURES[i] for i in idx]
    sorted_vals  = importances[idx]

    # Color: top-30% features in dark blue, rest in light blue
    threshold = np.percentile(importances, 70)
    colors    = ['#185FA5' if v >= threshold else '#93B8D9' for v in sorted_vals]

    fig, ax = plt.subplots(figsize=(10, max(6, len(BASE_FEATURES)*0.35)))
    ax.barh(sorted_feats[::-1], sorted_vals[::-1], color=colors[::-1])
    ax.set_xlabel('Feature Importance (Gain)', fontsize=11)
    ax.set_title(f'XGBoost Feature Importance — Active Users (H+24)',
                fontsize=12, fontweight='bold')
    ax.tick_params(axis='y', labelsize=9)
    plt.tight_layout()
    save_path = os.path.join(PLOT_DIR, 'fig_xgb_importance_ActiveUsers.png')
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Feature importance saved → {save_path}')
    print('\nTop-5 features:')
    for feat, val in zip(sorted_feats[:5], sorted_vals[:5]):
        print(f'  {feat:35s}  {val:.4f}')
else:
    print('XGBoost model not found — run section 7 first.')

## 13 · Summary

In [ ]:
print('\n' + '='*65)
print('  COMPARISON TABLE — Active Users')
print('='*65)
print(table.to_string())
print('='*65)
print(f'  Table → comparison_table_ActiveUsers.csv')
print(f'  Plots → {PLOT_DIR}/')
for f in sorted(os.listdir(PLOT_DIR)):
    print(f'    {f}')